## Lambda Authorizer Implementation

### Introduction: Securing Your API Endpoints

Welcome back! In the previous lesson, you learned how to keep your application's secrets safe using AWS Secrets Manager and IAM roles. Now, we are moving forward to another critical part of application security: protecting your API endpoints.

APIs are the doors to your application's data and features. If you leave them unprotected, anyone can access or misuse your resources. In this lesson, you will learn how to use a Lambda authorizer to control who can access your API Gateway endpoints. By the end, you will have built a real token-based authorization system and seen it in action.

---

## Quick Recall: API Gateway and Lambda Integration

Before we dive in, let's quickly remind ourselves how API Gateway and Lambda work together.

* **API Gateway** acts as the front door for your application. It receives HTTP requests from users.
* **Lambda functions** are small pieces of code that run in the cloud. API Gateway can send requests to Lambda functions, which then process the request and return a response.

For example, when someone visits your API endpoint, API Gateway forwards the request to your Lambda function, which handles the logic and sends back a result.

This flow is important to remember because we will be adding a security check (the Lambda authorizer) right in the middle of this process.

---

## What Is a Lambda Authorizer?

A Lambda authorizer is a special type of Lambda function that helps API Gateway decide if a request should be allowed or denied. It checks the incoming request for some kind of proof — usually a token — and then tells API Gateway if the request is authorized.

Here's how it works in simple terms:

1. A user sends a request to your API Gateway endpoint.
2. API Gateway calls your Lambda authorizer function first.
3. The authorizer checks for a valid token (like a password).
4. If the token is valid, the request is allowed to reach your protected Lambda function.
5. If the token is missing or incorrect, the request is denied.

A common way to send a token is in the `Authorization` header, like this:

```text
Authorization: Bearer letmein
```

In this lesson, you will build a Lambda authorizer that only accepts the token `letmein` and denies all other requests.

Here's a visual representation of this authorization flow:

```text
Client Request (Authorization: Bearer letmein)
       │
       ▼
  API Gateway
       │
       ▼
Lambda Authorizer ◄─── Checks token
       │
   ┌───┴───┐
   ▼       ▼
 Allow   Deny ──► 401/403
   │
   ▼
Protected Lambda ──► 200 Response
```

---

## Building the Lambda Authorizer Step-by-Step

Let's build the Lambda authorizer function together, step by step.

### 1. Extracting the Authorization Header

First, we need to get the `Authorization` header from the incoming event. The event is a Python dictionary that contains all the request details.

Here's how you can safely extract the header, handling both uppercase and lowercase keys:

```python
headers = event.get("headers", {}) or {}
auth_header = headers.get("Authorization") or headers.get("authorization")

# Log presence/format only - never log actual tokens in production!
if auth_header:
    header_format = auth_header.split()[0] if auth_header.split() else "invalid"
    print(f"Authorization header present with format: {header_format}")
else:
    print("Authorization header missing")
```

* `event.get("headers", {})` gets the headers dictionary from the event. If it's missing, it returns an empty dictionary.
* We check both `"Authorization"` and `"authorization"` to handle different clients.
* **Important:** We only log the presence and format (e.g., "Bearer") of the header, never the actual token value. Logging tokens is a security risk.

**Example Output:**

```text
Authorization header present with format: Bearer
```

### 2. Validating the Bearer Token

Now, let's check if the token is correct. We expect the header to be exactly `Bearer letmein`.

```python
EXPECTED_TOKEN = "letmein"

if auth_header == f"Bearer {EXPECTED_TOKEN}":
    effect = "Allow"
    principal_id = "user|valid"
    print("✅ Token is valid - allowing access")
else:
    effect = "Deny"
    principal_id = "user|anonymous"
    print("❌ Invalid or missing token - denying access")
```

> ⚠️ **Important Note About Hardcoding:** You might notice we're using a hardcoded token here (`EXPECTED_TOKEN = "letmein"`). In the previous lesson, you learned that hardcoding secrets is a security risk. You're right to question this! In a real production application, you would absolutely retrieve this token from AWS Secrets Manager using the techniques you learned in Lesson 1. However, for this lesson, we're keeping it simple to focus on understanding how Lambda authorizers work. In the upcoming practices, you'll combine both concepts — using Secrets Manager to retrieve tokens inside your authorizer function. Think of this as learning one concept at a time before putting them together.

* If the header matches, we set `effect` to `"Allow"` and mark the user as valid.
* Otherwise, we set `effect` to `"Deny"` and mark the user as anonymous.

**Example Output:**

```text
✅ Token is valid - allowing access
```

or

```text
❌ Invalid or missing token - denying access
```

### 3. Returning the IAM Policy

Finally, we need to return a policy document that tells API Gateway what to do.

```python
policy = {
    "principalId": principal_id,
    "policyDocument": {
        "Version": "2012-10-17",
        "Statement": [{
            "Action": "execute-api:Invoke",
            "Effect": effect,
            "Resource": event.get("methodArn", "*")
        }]
    }
}
print(f"Returning policy with effect: {effect}")
return policy
```

* The `policyDocument` tells API Gateway to allow or deny the request.
* The `principalId` is just an identifier for the user.

**Example Output:**

```text
Returning policy with effect: Allow
```

---

## Protecting and Responding from Your API Handler

Once your Lambda authorizer is set up, you can protect your API endpoint. API Gateway will only forward requests to your protected Lambda function if the authorizer allows it.

Here's a production-ready protected API handler that returns proper JSON:

```python
import json

def handler(event, context):
    response_body = {
        "message": "🎉 You are authorized!",
        "description": "This endpoint is protected by Lambda authorizer."
    }
    
    return {
        "statusCode": 200,
        "headers": {
            "Content-Type": "application/json"
        },
        "body": json.dumps(response_body)
    }
```

* If the request is authorized, this function returns a JSON response with proper `Content-Type` headers and status code 200.
* If the request is not authorized, API Gateway will block the request before it reaches this function.
* Always return JSON and set appropriate headers to follow production best practices.

**Example Output:**

```json
{
  "statusCode": 200,
  "headers": {
    "Content-Type": "application/json"
  },
  "body": "{\"message\": \"🎉 You are authorized!\", \"description\": \"This endpoint is protected by Lambda authorizer.\"}"
}
```

---

## Deploying and Testing Your Solution

Now that you have both the authorizer and the protected handler, it's time to deploy and test your solution.

1. **Deploying with AWS SAM:**

   Use the AWS SAM CLI to build and deploy your application. On CodeSignal, the required tools are already installed and configured, so you don't need to worry about setup here.

   > **Note:** When deploying to real AWS accounts, deployments create actual AWS resources (API Gateway, Lambda functions, IAM roles), typically take 2-5 minutes, and may incur small costs (usually under $1 for testing). Always clean up resources after testing to avoid ongoing charges. For local testing without AWS costs, you can use `sam local start-api` to test your functions locally.

2. **Testing Different Scenarios:**

   After deployment, you can test your API endpoint with different tokens:

   * **No Authorization header:** Access should be denied (401 or 403).
   * **Wrong token:** Access should be denied (401 or 403).
   * **Correct token** (`Bearer letmein`): Access should be allowed (200).

**Example Test Results:**

```text
🧪 Test 1: No Authorization Header (should return 401/403)
Status: 401
✅ PASS - Correctly denied access

🧪 Test 2: Wrong Token (should return 401/403)
Status: 403
✅ PASS - Correctly denied access

🧪 Test 3: Correct Token (should return 200)
Status: 200
✅ PASS - Successfully authorized
```

---

## Summary and What's Next

In this lesson, you learned how to secure your API Gateway endpoints using a Lambda authorizer. You built a function to check for a valid Bearer token, returned the correct IAM policy, and protected your API handler so only authorized requests get through. You also learned important security practices like never logging token values and returning properly formatted JSON responses with correct headers. Finally, you saw how to deploy and test your solution using AWS SAM.

Next, you'll get hands-on practice by completing the code and testing your own Lambda authorizer. This will help you reinforce what you've learned and prepare you for building more secure and robust APIs in the future. Good luck!

## Fix the Broken Authorization Header

> **Note:** the source pasted into this cell duplicated the entire lesson introduction above instead of a distinct exercise description. The cell has been trimmed to just the task and code — the actual bug below is real and matches the title, so the fix stands regardless.

You have a Lambda authorizer that only checks the capitalized `"Authorization"` key, but the lesson itself taught you to check **both** `"Authorization"` and `"authorization"`, since different clients and API Gateway payload formats can send the header in either case (API Gateway HTTP APIs, in particular, normalize all header names to lowercase). Your task is to fix `authorizer.py` so it looks up both variants again.

```python
import os
import json

EXPECTED_TOKEN = os.environ.get("EXPECTED_TOKEN", "letmein")

def handler(event, context):
    print(f"Authorizer received event: {json.dumps(event, indent=2)}")
    
    # Extract Authorization header
    headers = event.get("headers", {}) or {}
    auth_header = headers.get("Authorization")
    
    print(f"Authorization header: {auth_header}")
    
    # Determine if access should be allowed
    if auth_header == f"Bearer {EXPECTED_TOKEN}":
        effect = "Allow"
        principal_id = "user|valid"
        print("✅ Token is valid - allowing access")
    else:
        effect = "Deny" 
        principal_id = "user|anonymous"
        print("❌ Invalid or missing token - denying access")
    
    # Return IAM policy
    policy = {
        "principalId": principal_id,
        "policyDocument": {
            "Version": "2012-10-17",
            "Statement": [{
                "Action": "execute-api:Invoke",
                "Effect": effect,
                "Resource": event.get("methodArn", "*")
            }]
        }
    }
    
    print(f"Returning policy: {json.dumps(policy, indent=2)}")
    return policy
```

Here is the fixed `authorizer.py`, with the lowercase fallback restored:

```python
import os
import json

EXPECTED_TOKEN = os.environ.get("EXPECTED_TOKEN", "letmein")

def handler(event, context):
    print(f"Authorizer received event: {json.dumps(event, indent=2)}")
    
    # Extract Authorization header
    headers = event.get("headers", {}) or {}
    auth_header = headers.get("Authorization") or headers.get("authorization")
    
    print(f"Authorization header: {auth_header}")
    
    # Determine if access should be allowed
    if auth_header == f"Bearer {EXPECTED_TOKEN}":
        effect = "Allow"
        principal_id = "user|valid"
        print("✅ Token is valid - allowing access")
    else:
        effect = "Deny" 
        principal_id = "user|anonymous"
        print("❌ Invalid or missing token - denying access")
    
    # Return IAM policy
    policy = {
        "principalId": principal_id,
        "policyDocument": {
            "Version": "2012-10-17",
            "Statement": [{
                "Action": "execute-api:Invoke",
                "Effect": effect,
                "Resource": event.get("methodArn", "*")
            }]
        }
    }
    
    print(f"Returning policy: {json.dumps(policy, indent=2)}")
    return policy
```

## Complete the Token Validation Logic

## Complete the IAM Policy Structure

## Enhance API Response with Context Data

## Add Error Handling and Edge Cases